# Iowa Bridge Spatial Autocorrelation Lab

This is a standalone Python analysis of **IowaCounty.json** and **bridgedata.json** in this folder. It follows the course workflow with GeoPandas, libpysal, and esda. It rebuilds weights from the source geometries and coordinates; it does not read the website's JavaScript or precomputed weights file.

Choose a scale in the controls:

- **Statewide county summaries**: all 99 county records, with Queen and Rook contiguity. The selected measure is compared across both weights. Sensitivity mode also reports all planned county measures.
- **Selected-county bridge points**: records filtered by the bridge file's `brcounty` field, with K-nearest-neighbor weights rebuilt from projected coordinates.

Each run draws a Moran scatterplot with a fitted slope and a permutation distribution with the observed statistic marked. All observations remain in the plotted range; the five most extreme standardized values are labeled.

County rates and bridge-level indicators use different spatial units. Flood closures are modeled scenarios, not observed bridge failures. Detour length is omitted because code 99 still needs source interpretation.

If the course environment lacks packages, run `%pip install geopandas libpysal esda matplotlib ipywidgets` in Jupyter and restart the kernel.

In [ ]:
from pathlib import Path
import copy
import geopandas as gpd
import libpysal
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display, Markdown
from esda.moran import Moran
from libpysal.weights import Queen, Rook, KNN
from libpysal.weights.spatial_lag import lag_spatial

# Find the original inputs from this notebook folder or its parent.
roots = [Path.cwd(), Path.cwd() / "final-project", *Path.cwd().parents]
PROJECT_DIR = next((p for p in roots if (p / "IowaCounty.json").exists()
                    and (p / "bridgedata.json").exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from final-project or its parent.")

COUNTY_FILE, BRIDGE_FILE = PROJECT_DIR / "IowaCounty.json", PROJECT_DIR / "bridgedata.json"
ANALYSIS_CRS = "EPSG:26915"
DEFAULT_PERMUTATIONS, DEFAULT_SEED = 999, 54321

plt.rcParams.update({
    "figure.facecolor": "#f4f1ec", "axes.facecolor": "#ffffff",
    "axes.edgecolor": "#98a2b3", "axes.labelcolor": "#344054",
    "text.color": "#1f2933", "xtick.color": "#475467",
    "ytick.color": "#475467", "font.family": "DejaVu Sans",
    "axes.titleweight": "semibold", "axes.grid": False,
})
print(f"Project folder: {PROJECT_DIR}")
print(f"libpysal {libpysal.__version__}; GeoPandas {gpd.__version__}")

In [ ]:
# Load the same source GeoJSON files. KNN distances use projected meters.
counties = gpd.read_file(COUNTY_FILE)
bridges = gpd.read_file(BRIDGE_FILE)
counties = counties.set_crs(ANALYSIS_CRS) if counties.crs is None else counties.to_crs(ANALYSIS_CRS)
bridges = bridges.set_crs(ANALYSIS_CRS) if bridges.crs is None else bridges.to_crs(ANALYSIS_CRS)

counties["FIPS"] = counties["FIPS"].astype(str).str.zfill(5)
counties["CountyName"] = counties["CountyName"].astype(str).str.strip()
bridges["brcounty"] = bridges["brcounty"].astype(str).str.strip()
bridges["brid"] = bridges["brid"].astype(str)
if counties["FIPS"].duplicated().any() or bridges["brid"].duplicated().any():
    raise ValueError("County FIPS and bridge identifiers must be unique.")

# Check brcounty record counts against the county summaries.
counts = bridges.groupby("brcounty").size()
county_check = counties[["FIPS", "CountyName", "Bridge"]].copy()
county_check["records_by_brcounty"] = county_check["CountyName"].map(counts).fillna(0).astype(int)
county_check["difference"] = county_check["records_by_brcounty"] - pd.to_numeric(county_check["Bridge"])
display(county_check.loc[county_check["difference"].ne(0)])
print(f"{len(counties):,} counties; {len(bridges):,} bridge points; "
      f"{bridges['brcounty'].nunique():,} brcounty values; "
      f"{int(county_check['difference'].ne(0).sum())} count mismatches.")

counties = counties.set_index("FIPS", drop=False).sort_index()
bridges = bridges.set_index("brid", drop=False).sort_index()

COUNTY_VARIABLES = {
    "br_poorper": "Poor-condition bridge percentage",
    "fd_50per": "50-year modeled closure percentage",
    "fd_100per": "100-year modeled closure percentage",
    "fd_500per": "500-year modeled closure percentage",
    "br_age": "Average bridge age (years)",
}
BRIDGE_VARIABLES = {
    "poor_flag": "Poor-condition indicator (Poor = 1)",
    "brage": "Bridge age (years)",
    "bradt": "Average daily traffic",
    "bradt_log1p": "Log (1 + average daily traffic)",
    "closure_severity": "Modeled closure severity (0-3 scenarios)",
    "br50": "50-year modeled closure indicator",
    "br100": "100-year modeled closure indicator",
    "br500": "500-year modeled closure indicator",
}

def bridge_measure(frame, variable):
    """Map documented bridge fields to numeric analysis variables."""
    if variable == "poor_flag":
        return frame["brcond"].eq("Poor").astype(float)
    if variable == "closure_severity":
        return frame[["br50", "br100", "br500"]].fillna(False).astype(bool).sum(axis=1).astype(float)
    if variable in {"br50", "br100", "br500"}:
        return frame[variable].fillna(False).astype(bool).astype(float)
    if variable == "bradt_log1p":
        return np.log1p(pd.to_numeric(frame["bradt"], errors="coerce"))
    if variable in {"brage", "bradt"}:
        return pd.to_numeric(frame[variable], errors="coerce")
    raise KeyError(f"Unsupported bridge measure: {variable}")

In [ ]:
def build_county_weights(kind):
    """Rebuild Queen/Rook contiguity from county polygons."""
    ids = counties.index.astype(str).tolist()
    w = {"Queen": Queen, "Rook": Rook}[kind].from_dataframe(counties, ids=ids)
    w.id_order = ids
    if w.islands:
        raise ValueError(f"{kind} weights contain islands: {w.islands}")
    if w.n != len(counties):
        raise ValueError(f"{kind} weights do not cover all county records.")
    w.transform = "R"
    return w


def build_bridge_knn_weights(frame, k):
    """Build symmetric KNN weights from projected bridge points."""
    ids = frame.index.astype(str).tolist()
    if len(ids) <= k:
        raise ValueError(f"Need more than {k} bridges for KNN with k={k}.")
    w = KNN.from_dataframe(frame, k=k, ids=ids).symmetrize()
    w.id_order = ids
    if w.islands:
        raise ValueError(f"KNN(k={k}) contains islands: {w.islands}")
    w.transform = "R"
    return w


def moran_run(frame, values, weights, *, label, unit, permutations, seed):
    """Run esda.Moran and compare I with the direct row-standardized matrix formula."""
    w = copy.deepcopy(weights)
    w.transform = "R"
    y = pd.Series(np.asarray(values, dtype=float), index=frame.index).reindex(w.id_order)
    if y.isna().any() or not np.isfinite(y.to_numpy()).all():
        raise ValueError(f"{label} has missing or non-finite observations.")
    if y.nunique() < 2:
        raise ValueError(f"{label} is constant in this analysis scope.")

    # Use the course's fixed NumPy seed pattern for reproducible permutations.
    np.random.seed(int(seed))
    mi = Moran(y.to_numpy(), w, transformation="R", permutations=int(permutations))
    z = np.asarray(mi.z, dtype=float)
    direct_i = len(y) / mi.w.s0 * float(z @ (mi.w.sparse @ z)) / float(z @ z)
    difference = abs(float(mi.I) - direct_i)
    if difference > 1e-10:
        raise ArithmeticError(f"esda/direct-matrix I differs by {difference:.3g}.")

    sims = np.asarray(mi.sim, dtype=float)
    # Pre-specified positive upper-tail pseudo-p, plus-one correction.
    p_upper = (int(np.count_nonzero(sims >= float(mi.I))) + 1) / (len(sims) + 1)
    lag = np.asarray(lag_spatial(mi.w, z), dtype=float)
    lag -= lag.mean()
    ordered = frame.loc[mi.w.id_order]
    ids = ordered.index.astype(str).to_numpy()
    names = (ordered["CountyName"].astype(str).to_numpy()
             if "CountyName" in ordered.columns
             else np.array([f"Bridge {v}" for v in ordered["brid"].astype(str)]))
    return {
        "I": float(mi.I), "manual_I": float(direct_i), "I_difference": difference,
        "p_upper": float(p_upper), "p_min": 1 / (len(sims) + 1),
        "n": len(y), "w": mi.w,
        "pairs": sum(map(len, mi.w.neighbors.values())) // 2,
        "sims": sims, "x": z, "lag": lag, "ids": ids, "names": names,
        "label": label, "unit": unit, "permutations": len(sims), "seed": int(seed),
    }


def padded_limits(values, fraction=0.08):
    low, high = float(np.min(values)), float(np.max(values))
    pad = max((high - low) * fraction, 0.08)
    return low - pad, high + pad


def plot_scatter(ax, result, title):
    x, y = result["x"], result["lag"]
    extreme = np.argsort(np.maximum(np.abs(x), np.abs(y)))[-min(5, len(x)):]
    colors = np.full(len(x), "#2f5d7c", dtype=object)
    colors[extreme] = "#e07a3f"
    ax.scatter(x, y, s=34 if len(x) < 150 else 15, c=colors, alpha=.78,
               edgecolors="white", linewidths=.35, zorder=3)
    xlim, ylim = padded_limits(x), padded_limits(y)
    line_x = np.linspace(*xlim, 100)
    ax.plot(line_x, result["I"] * line_x, color="#b42318", linestyle="--",
            linewidth=1.8, label=f"Fitted slope = I ({result['I']:.4f})")
    ax.axhline(0, color="#667085", linestyle=":", linewidth=.9)
    ax.axvline(0, color="#667085", linestyle=":", linewidth=.9)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_xlabel("Standardized value"); ax.set_ylabel("Centered spatial lag")
    ax.set_title(title); ax.legend(frameon=False, fontsize=8)
    for i in extreme:
        ax.annotate(str(result["names"][i]), (x[i], y[i]), xytext=(4,4),
                    textcoords="offset points", fontsize=7, color="#344054")


def plot_permutations(ax, result, title):
    ax.hist(result["sims"], bins=25, color="#8fb3c9", edgecolor="white", linewidth=.6)
    ax.axvline(result["I"], color="#b42318", linewidth=2,
               label=f"Observed I = {result['I']:.4f}")
    ax.set_xlabel("Simulated Moran's I"); ax.set_ylabel("Permutation count")
    ax.set_title(title); ax.legend(frameon=False, fontsize=8)


def plot_diagnostics(results, title):
    fig, axes = plt.subplots(2, len(results), figsize=(6.2*len(results), 8.2),
                             squeeze=False, gridspec_kw={"height_ratios":[1,.82]},
                             constrained_layout=True)
    for col, (name, result) in enumerate(results):
        plot_scatter(axes[0,col], result, f"{name}: Moran scatterplot")
        plot_permutations(axes[1,col], result, f"{name}: permutation distribution")
    fig.suptitle(title, fontsize=14, fontweight="semibold")
    plt.show()


def result_row(result, measure, weights_name):
    return {
        "Measure": measure, "Weights": weights_name, "n": result["n"],
        "Neighbor pairs": result["pairs"], "Moran I (esda)": result["I"],
        "Moran I (matrix check)": result["manual_I"],
        "Absolute I difference": result["I_difference"],
        "Upper-tail pseudo-p": result["p_upper"], "Minimum p": result["p_min"],
        "Permutations": result["permutations"], "Seed": result["seed"],
    }

In [ ]:
def analyze_statewide(variable, permutations, seed, compare_all=True):
    """Analyze county summaries with Queen/Rook comparison."""
    measures = list(COUNTY_VARIABLES) if compare_all else [variable]
    weights_by_name = {name: build_county_weights(name) for name in ("Queen", "Rook")}
    rows, selected = [], []
    for key in measures:
        values = pd.to_numeric(counties[key], errors="coerce")
        for weights_name, w in weights_by_name.items():
            result = moran_run(
                counties, values, w, label=COUNTY_VARIABLES[key],
                unit="Iowa county", permutations=permutations, seed=seed,
            )
            rows.append(result_row(result, key, weights_name))
            if key == variable:
                selected.append((weights_name, result))
    display(pd.DataFrame(rows).round(6))
    plot_diagnostics(selected, f"Statewide county summaries · {COUNTY_VARIABLES[variable]}")
    display(Markdown(
        "County rates have different denominators. Global Moran's I describes "
        "statewide association; it does not identify county hotspots."
    ))


def analyze_selected_county(county_name, variable, k, permutations, seed, compare_k=True):
    """Filter bridge records using brcounty, then analyze only this county."""
    selected = bridges.loc[bridges["brcounty"].eq(county_name)].copy()
    if selected.empty:
        raise ValueError(f"No records matched brcounty == {county_name!r}.")
    expected = int(pd.to_numeric(
        counties.loc[counties["CountyName"].eq(county_name), "Bridge"]
    ).iloc[0])
    if len(selected) != expected:
        raise ValueError(
            f"{county_name}: brcounty matched {len(selected)}, summary reports {expected}."
        )

    values = bridge_measure(selected, variable)
    k_values = [4, 6, 8, 12] if compare_k else [int(k)]
    if int(k) not in k_values:
        k_values.append(int(k))
    rows, selected_result = [], None
    for current_k in sorted(set(k_values)):
        w = build_bridge_knn_weights(selected, current_k)
        result = moran_run(
            selected, values, w, label=BRIDGE_VARIABLES[variable],
            unit=f"{county_name} County bridge", permutations=permutations, seed=seed,
        )
        rows.append(result_row(result, BRIDGE_VARIABLES[variable], f"KNN union, k={current_k}"))
        if current_k == int(k):
            selected_result = result

    print(f"{county_name} County: {len(selected):,} records matched by brcounty; summary = {expected:,}.")
    display(pd.DataFrame(rows).round(6))
    plot_diagnostics([(f"KNN k={k}", selected_result)],
                     f"{county_name} County bridge scale · {BRIDGE_VARIABLES[variable]}")
    display(Markdown(
        "Bridge KNN uses projected EPSG:26915 coordinates; the graph is "
        "symmetrized and row-standardized. This is within-county analysis, not "
        "county-to-county analysis. Log(1 + traffic) helps inspect right-skew."
    ))

In [ ]:
scale_widget = widgets.ToggleButtons(
    options=[("Statewide county summaries", "state"),
             ("Selected-county bridge points", "county")],
    value="state", description="Scale:",
)
county_widget = widgets.Dropdown(
    options=sorted(counties["CountyName"].unique()),
    value=sorted(counties["CountyName"].unique())[0], description="County:",
)
variable_widget = widgets.Dropdown(
    options=[(label, key) for key, label in COUNTY_VARIABLES.items()],
    value="br_poorper", description="Measure:",
)
neighbors_widget = widgets.Dropdown(options=[4, 6, 8, 12], value=8, description="K neighbors:")
permutations_widget = widgets.IntText(value=DEFAULT_PERMUTATIONS, description="Permutations:")
seed_widget = widgets.IntText(value=DEFAULT_SEED, description="Seed:")
sensitivity_widget = widgets.Checkbox(value=True, description="Compare all planned county variables", indent=False)
run_button = widgets.Button(description="Run Moran analysis", button_style="primary", icon="bar-chart")
output = widgets.Output()


def update_controls(*_):
    county_mode = scale_widget.value == "county"
    county_widget.layout.display = "" if county_mode else "none"
    neighbors_widget.layout.display = "" if county_mode else "none"
    current = variable_widget.value
    if county_mode:
        options, default = [(v, k) for k, v in BRIDGE_VARIABLES.items()], "poor_flag"
        sensitivity_widget.description = "Compare KNN k = 4, 6, 8, 12"
    else:
        options, default = [(v, k) for k, v in COUNTY_VARIABLES.items()], "br_poorper"
        sensitivity_widget.description = "Compare all planned county variables"
    variable_widget.options = options
    variable_widget.value = current if current in {k for _, k in options} else default


def run_selected_analysis(_):
    with output:
        clear_output(wait=True)
        try:
            if permutations_widget.value < 99:
                raise ValueError("Use at least 99 permutations.")
            if seed_widget.value < 0:
                raise ValueError("Seed must be zero or positive.")
            if scale_widget.value == "state":
                analyze_statewide(variable_widget.value, permutations_widget.value,
                                  seed_widget.value, sensitivity_widget.value)
            else:
                analyze_selected_county(county_widget.value, variable_widget.value,
                                        neighbors_widget.value, permutations_widget.value,
                                        seed_widget.value, sensitivity_widget.value)
        except Exception as error:
            display(Markdown(f"**Analysis could not run:** {error}"))


scale_widget.observe(update_controls, names="value")
run_button.on_click(run_selected_analysis)
update_controls()
display(widgets.VBox([
    scale_widget, county_widget, variable_widget, neighbors_widget,
    permutations_widget, seed_widget, sensitivity_widget, run_button,
]), output)